# Train parasite-egg detector (Chula-ParasiteEgg-11) → ONNX
Runtime → Change runtime type → **T4 GPU** (or better).

In [3]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')  # checkpoints + final model are saved here

Wed Nov  4 09:46:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Private repo: add a GitHub token as Colab Secret GH_TOKEN (key icon in the left sidebar)
from google.colab import userdata
try:
    tok = userdata.get("GH_TOKEN")
except Exception:
    tok = None
url = f"https://{tok}@github.com/rtpsticky/ParasiteEgg.git" if tok else "https://github.com/rtpsticky/ParasiteEgg.git"
%cd /content
!rm -rf ParasiteEgg && git clone -q {url}
%cd /content/ParasiteEgg
# checkpoints live on Drive so they survive the session
!mkdir -p /content/drive/MyDrive/parasite_runs && ln -sfn /content/drive/MyDrive/parasite_runs runs
!pip -q install -r requirements.txt

/content
/content/ParasiteEgg


## Download from Zenodo (record 15866298, MIT license)

In [ ]:
%%bash
# The converted dataset (~3 GB) is cached on Drive: the first session downloads + converts,
# later sessions just copy the cache (a few minutes instead of 30+).
set -e
CACHE=/content/drive/MyDrive/parasite_data/yolo.tar
if [ -f "$CACHE" ]; then
  echo "using cached dataset $CACHE"
  tar -xf "$CACHE" -C /content
else
  mkdir -p /content/raw && cd /content/raw
  B=https://zenodo.org/records/15866298/files
  wget -q --show-progress -c "$B/Chula-ParasiteEgg-11.zip?download=1" -O train.zip
  wget -q --show-progress -c "$B/Chula-ParasiteEgg-11_test.zip?download=1" -O test.zip
  wget -q -c "$B/test_labels_200.json?download=1" -O test_labels_200.json
  echo "1621521969547b0291c0cfbf7e450074  train.zip" | md5sum -c
  echo "a47fdab77ced45bc40b0e8b1e6ff8d22  test.zip" | md5sum -c
  unzip -q train.zip -d train && rm train.zip
  unzip -q test.zip -d test && rm test.zip
  cd /content/ParasiteEgg
  python scripts/prepare_dataset.py --src /content/raw/train --out /content/yolo      --test-src /content/raw/test --test-json /content/raw/test_labels_200.json --workers 8
  rm -rf /content/raw
  mkdir -p "$(dirname "$CACHE")" && tar -cf "$CACHE" -C /content yolo
fi
cat /content/yolo/data.yaml


## Train
`yolo11n` is small and fast in the browser. Try `yolo11s.pt` if accuracy isn't enough. After a disconnect, re-run download/prepare, then the resume cell.

In [ ]:
!python scripts/train.py --data /content/yolo/data.yaml --model yolo11n.pt --epochs 100 --device 0

## Session ran out? Check progress, then resume
Re-run the cells above (except training), then:

In [ ]:
import pandas as pd
r = pd.read_csv("runs/detect/parasite/results.csv"); r.columns = r.columns.str.strip()
print(r[["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(15).to_string(index=False))


In [ ]:
# continue from last.pt (keeps optimizer + LR schedule, finishes at epoch 100)
!python scripts/train.py --resume


## Evaluate on test + export ONNX

In [ ]:
!python scripts/export_onnx.py --weights runs/detect/parasite/weights/best.pt --data /content/yolo/data.yaml --out /content/drive/MyDrive/parasite_runs/web_model
!ls -la /content/drive/MyDrive/parasite_runs/web_model